# Plug-in architecture, manifest-first: interactive walkthrough

A radiology report workbench whose plugins are folders. Each folder has a `plugin.toml` (data) and, optionally, Python (code).
The host reads every manifest at startup, and imports a plugin's code only when one of its contributions is first used.

Each section below runs one stage on its own. The [README](README.md) has the full story; `demo.py` runs the same stages as six scenes.

In [1]:
import sys
import tempfile
from dataclasses import fields
from pathlib import Path

from host import EXTENSION_POINTS, MODULE_PREFIX, ManifestError, discover, parse_manifest

PLUGINS = Path("plugins")          # three healthy plugins
BROKEN = Path("plugins-broken")    # one failure each


def imported() -> list[str]:
    """Ground truth: which plugin modules has Python actually imported?"""
    return sorted(m for m in sys.modules if m.startswith(MODULE_PREFIX))

## 1. A manifest is data

The whole contract a plugin offers sits in one TOML file. The `handler` is a string, `"module:function"`: a pointer the host follows later, not an import.

In [2]:
print((PLUGINS / "measure" / "plugin.toml").read_text())

id = "measure"
version = "0.1.0"
api = 1                              # the host contract this plugin was built for

[[contributes.commands]]
id = "measure.to-cm"
title = "Convert mm to cm"
handler = "measure:to_cm"            # resolved on first run, not at startup



In [3]:
parse_manifest(PLUGINS / "measure")

Manifest(id='measure', version='0.1.0', api=1, folder=PosixPath('plugins/measure'), contributes={'commands': [Command(id='measure.to-cm', title='Convert mm to cm', handler='measure:to_cm')]})

The host owns the extension points. A plugin can only fill these slots, and each slot's dataclass fields are the keys a manifest must give.

In [4]:
{point: [f.name for f in fields(schema)] for point, schema in EXTENSION_POINTS.items()}

{'commands': ['id', 'title', 'handler'], 'templates': ['id', 'title', 'body']}

## 2. The gate, one manifest at a time

`parse_manifest()` checks each manifest alone: required keys, types, the api version, known extension points.
Two broken plugins fail here. The other two pass, because their faults are invisible to a single manifest:
`measure-fork` collides with *another* plugin, and `impression-tools` disagrees with its *own code*.

In [5]:
for folder in sorted(BROKEN.iterdir()):
    try:
        parse_manifest(folder)
        print(f"passes   {folder.name}")
    except ManifestError as e:
        print(f"rejected {folder.name:<18} {e}")

passes   impression-tools
passes   measure-fork
rejected mri-knee-template  contributes.templates[0].title: required
rejected too-new            api: built for host api 2, this host is 1


## 3. Discovery: the whole system, from data

`discover()` parses every manifest and then checks that every contribution id has one owner.
The palette (what a command menu or `--help` would show) comes from data alone. Nothing is imported.

In [6]:
reg = discover(PLUGINS)
for point, contribution, owner in reg.palette():
    print(f"{point:<10} {contribution.id:<20} {contribution.title:<24} {owner}")
imported()

commands   measure.to-cm        Convert mm to cm         measure
commands   report.word-count    Count words              report-tools
templates  ct-brain             CT brain, non-contrast   ct-brain-template


[]

## 4. Lazy activation

The first `run()` follows the handler string: `[activate]` marks the import. A second run reuses the cached module.

In [7]:
reg.run("report.word-count", "No acute intracranial abnormality.")

  [activate] report-tools ← report-tools/report_tools.py


'4 words'

In [8]:
print(reg.run("report.word-count", "Gray-white matter differentiation is preserved."))
imported()

5 words


['plugins.report-tools.report_tools']

## 5. Conflicts and drift

Add the broken plugins. The per-manifest gate rejects two of them, and the whole-system check rejects both claimants of `measure.to-cm`.
The drift in `impression-tools` passes discovery. `check()`, the contract test, finds it, at the cost of importing every code plugin.

In [9]:
reg_all = discover(PLUGINS, BROKEN)
reg_all.rejected

{'mri-knee-template': 'contributes.templates[0].title: required',
 'too-new': 'api: built for host api 2, this host is 1',
 'measure': "command 'measure.to-cm' is also claimed by measure-fork",
 'measure-fork': "command 'measure.to-cm' is also claimed by measure"}

In [10]:
reg_all.check()

  [activate] impression-tools ← impression-tools/impression_tools.py


["impression.uppercase: impression-tools: manifest names 'impression_tools:uppercase', but impression_tools.py has no uppercase()"]

## 6. The seam test: add a plugin without touching `host.py`

Write a new plugin folder somewhere else and point discovery at it. `host.py` does not change.

In [11]:
extra = Path(tempfile.mkdtemp())
(extra / "measure-inch").mkdir()
(extra / "measure-inch" / "plugin.toml").write_text("""\
id = "measure-inch"
version = "0.1.0"
api = 1

[[contributes.commands]]
id = "measure.to-inch"
title = "Convert mm to inches"
handler = "inch:to_inch"
""")
(extra / "measure-inch" / "inch.py").write_text("""\
import re


def to_inch(text: str) -> str:
    return re.sub(r"(\\d+(?:\\.\\d+)?)\\s*mm\\b", lambda m: f"{float(m[1]) / 25.4:.2f} in", text)
""")

reg = discover(PLUGINS, extra)
reg.run("measure.to-inch", "A 12 mm nodule.")

  [activate] measure-inch ← measure-inch/inch.py


'A 0.47 in nodule.'

Adding a new *extension point* (say, `checks`) is different: it means a new dataclass in `EXTENSION_POINTS`, and code in the host that uses it.
That is by design. The host owns the slots; plugins fill them. More to try: the Exercises in the [README](README.md).